# CS 673 – Graph Database Final Project  
## NYC 311 Complaints Network Analysis  



**Semester:** Fall 2025


## 1. Problem Definition

### Project Title  
NYC 311 Service Requests – Graph Database Analysis

### Problem Statement  
New York City’s 311 system receives a high volume of service requests related to public services such as noise complaints, sanitation issues, housing conditions, and infrastructure concerns. Each service request is associated with multiple entities, including the complaint type, the agency responsible for handling the request, the geographic location where the issue occurred, and the request’s current status.

Traditional relational databases store this information in tables, which can make it difficult to efficiently explore complex relationships among these entities. For example, understanding how specific complaint types are distributed across boroughs or how agencies interact with different categories of complaints requires multiple joins and complex queries.

### Objective  
The objective of this project is to design and implement a graph database using NYC 311 service request data in order to model and analyze the relationships between complaints, agencies, locations, and statuses. By representing these entities as nodes and their connections as relationships, the graph database enables more intuitive and efficient querying of interconnected data.

### Key Questions  
This graph database is designed to answer questions such as:  
- Which complaint types occur most frequently in each borough?  
- Which agencies handle the largest number of service requests?  
- Which complaint types are most commonly associated with unresolved or open requests?  
- How are complaint patterns distributed across different locations within the city?

### Dataset Description  
The dataset used in this project is the NYC 311 Service Requests dataset obtained from NYC Open Data. To ensure efficient processing and analysis, the dataset is filtered to include a specific time range and a subset of relevant attributes such as complaint type, agency, borough, location, and request status.

### Expected Outcome  
By modeling NYC 311 service requests as a graph, this project demonstrates how graph databases can effectively represent and analyze highly connected urban service data. The resulting graph structure supports aggregation, relationship-based queries, and insight discovery that are difficult to achieve using traditional tabular data models.


## 2. Graph Database Justification

### Why a Graph Database?
The NYC 311 Service Requests dataset contains highly interconnected data. Each service request is linked to multiple entities, including the type of complaint, the agency responsible for handling the request, the geographic location, and the current request status. These entities naturally form a network of relationships rather than independent records.

A graph database is well suited for this problem because it stores data as nodes and relationships, allowing direct traversal between connected entities. This structure makes it efficient to explore patterns such as how complaint types relate to specific boroughs or how agencies interact with different categories of service requests.

### Limitations of Relational Databases
In a traditional relational database, analyzing relationships among complaints, locations, agencies, and statuses requires multiple join operations across tables. As the number of relationships increases, these queries become more complex and harder to optimize. Additionally, discovering indirect relationships (for example, complaint types frequently associated with the same agency and borough) is not intuitive in a tabular model.

### Advantages of a Graph Model for NYC 311 Data
Using a graph database provides the following advantages:
- Direct representation of relationships between service requests, complaint types, agencies, and locations  
- Efficient traversal of connected data without costly join operations  
- Flexible schema that allows new node types or relationships to be added easily  
- Natural support for aggregation and pattern-based queries  

### Conclusion
Given the highly relational nature of NYC 311 service request data, a graph database provides a more intuitive and efficient data model than a relational database. It enables meaningful analysis of complaint patterns, agency workload, and geographic trends through relationship-based queries and aggregations.


## 3. Graph Database Modeling

### Overview
The NYC 311 dataset contains service requests that connect to multiple entities (complaint type, agency, location, and status). To represent these real-world relationships clearly, we model the data as a property graph where each real entity is a node and each connection is a relationship.

---

### Node Types and Key Properties

**1) `Request` (individual 311 service request)**
- `unique_key` (primary identifier)
- `created_date`
- `closed_date` (may be null)
- `resolution_description` (text, may be null)

**2) `ComplaintType`**
- `name` (e.g., "Noise - Residential", "Illegal Parking")

**3) `Agency`**
- `name` (e.g., "NYPD", "DSNY", "HPD")

**4) `Borough`**
- `name` (e.g., "MANHATTAN", "BROOKLYN")

**5) `Status`**
- `name` (e.g., "Open", "Closed", "Pending")

*(Optional, if included in our cleaned dataset)*
**6) `ZipCode`**
- `zip` (e.g., "10027")

---

### Relationship Types

**Core relationships**
- `(Request)-[:HAS_COMPLAINT_TYPE]->(ComplaintType)`
- `(Request)-[:HANDLED_BY]->(Agency)`
- `(Request)-[:IN_BOROUGH]->(Borough)`
- `(Request)-[:HAS_STATUS]->(Status)`

*(Optional, if ZipCode is included)*
- `(Request)-[:IN_ZIP]->(ZipCode)`

---

### Modeling Assumptions
- Each `Request` has exactly one complaint type, one agency, one status, and one borough after cleaning.
- Null or missing values are handled during preprocessing so that nodes and relationships are created consistently.
- `unique_key` is used to prevent duplicate `Request` nodes.
- Text fields (complaint type, agency, borough, status) are standardized (trimmed, consistent casing) to reduce duplicates.

This schema follows a property graph model where nodes represent real-world entities and relationships represent interactions between them.


## 4. Database Connection Setup (Neo4j Aura)

### Objective
The objective of this step is to establish a secure connection between the Google Colab environment and a Neo4j Aura graph database instance. This connection enables programmatic creation of nodes, relationships, and execution of Cypher queries directly from Python.

### Database Environment
This project uses **Neo4j Aura**, a fully managed cloud-based graph database service. Neo4j Aura provides a secure and scalable environment for storing and querying graph data without requiring local installation or manual database configuration.

### Connection Method
The connection is established using the official **Neo4j Python Driver**. The driver requires:
- A secure connection URI (`neo4j+s://`)
- A valid database username
- A password associated with the Aura instance

To protect sensitive credentials, environment variables are used to store the connection details instead of hardcoding them directly in the notebook.

### Verification
After creating the driver instance, the connection is verified using Neo4j’s `verify_connectivity()` method. A successful verification confirms that the database is reachable and ready to accept queries and data insertion commands.

### Outcome
Upon successful execution of the connection code, the Google Colab notebook is fully connected to the Neo4j Aura database and ready for graph creation, data loading, querying, and aggregation operations.


In [6]:
!pip -q install neo4j pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 8.5 MB/s eta 0:00:00


In [2]:
import os
from getpass import getpass

# Neo4j Aura connection info (paste from AuraDB connection details)
os.environ["NEO4J_URI"] = input("Enter NEO4J_URI (e.g., neo4j+s://xxxx.databases.neo4j.io): ").strip()
os.environ["NEO4J_USER"] = input("Enter NEO4J_USER (usually neo4j): ").strip()
os.environ["NEO4J_PASSWORD"] = getpass("Enter NEO4J_PASSWORD: ")


Enter NEO4J_URI (e.g., neo4j+s://xxxx.databases.neo4j.io): neo4j+s://8454b2ea.databases.neo4j.io
Enter NEO4J_USER (usually neo4j): neo4j
Enter NEO4J_PASSWORD: ··········


In [7]:
from neo4j import GraphDatabase
import os

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"])
)

driver.verify_connectivity()
print("Connection successful")


Connection successful


## 5. Data Loading, Cleansing, and Transformation (NYC 311)

### Purpose
Before creating nodes and relationships in Neo4j, the NYC 311 dataset must be prepared to ensure consistency and to avoid duplicate entities. This step focuses on:
- Selecting only relevant columns required for the graph model  
- Handling missing (null) values in key fields  
- Standardizing text values to reduce duplicates (e.g., inconsistent casing or extra spaces)  
- Converting date/time columns to proper datetime format for later analysis and aggregation  
- Reducing dataset size to a manageable subset for efficient loading into Neo4j  

### Cleaning Strategy
The graph requires certain fields to exist for each request (Unique Key, Complaint Type, Agency, Borough, Status). Rows missing any of these critical values are removed. Non-critical fields such as Closed Date may be missing and are handled safely.

### Transformation Strategy
Text columns are normalized (trimmed and consistent casing). Date fields are converted into datetime to support time-based aggregation queries (e.g., requests per month or resolution time).


In [12]:
import pandas as pd

# Load NYC 311 dataset
df = pd.read_csv(
    "/content/311_Service_Requests_from_2010_to_Present_20251215.csv",
    engine='python', # Added 'engine="python"' to handle potential parsing issues
    on_bad_lines='skip' # Skip malformed lines
)

print(df.shape)
df.head()

(140388, 41)


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Complaint Type,Descriptor,Location Type,Incident Zip,Incident Address,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
0,60739073,03/31/2024 07:13:48 PM,04/09/2024 11:30:40 AM,EDC,Economic Development Corporation,Noise - Helicopter,Other,Above Address,10023.0,10 WEST 75 STREET,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.778408,-73.975130,"(40.778408491033055, -73.97512950765011)"
1,60733861,03/31/2024 07:13:40 PM,03/31/2024 08:03:56 PM,NYPD,New York City Police Department,Illegal Parking,Blocked Sidewalk,Street/Sidewalk,10462.0,1833 HUNT AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.846254,-73.865509,"(40.84625364665493, -73.8655088221215)"
2,60736342,03/31/2024 07:13:27 PM,04/01/2024 08:53:05 AM,TLC,Taxi and Limousine Commission,Lost Property,Electronics/Phones,Taxi,10013.0,46 BOWERY,...,NaN,NaN,"46 BOWERY, MANHATTAN (NEW YORK), NY, 10013",NaN,NaN,NaN,NaN,40.715696,-73.996414,"(40.715696356000116, -73.99641434248574)"
3,60739177,03/31/2024 07:13:22 PM,03/31/2024 07:28:24 PM,NYPD,New York City Police Department,Illegal Parking,Blocked Crosswalk,Street/Sidewalk,11235.0,2601 VOORHIES AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.586916,-73.943050,"(40.586916344525434, -73.94304973805869)"
4,60738322,03/31/2024 07:13:20 PM,03/31/2024 07:38:53 PM,NYPD,New York City Police Department,Illegal Parking,Blocked Hydrant,Street/Sidewalk,10025.0,57 WEST 94 STREET,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.791161,-73.967497,"(40.79116136842613, -73.96749748697144)"


In [14]:
columns_to_keep = [
    "Unique Key",
    "Created Date",
    "Closed Date",
    "Agency",
    "Agency Name",
    "Complaint Type",
    "Borough",
    "Status"
]

df = df[columns_to_keep].copy()
print("After column selection:", df.shape)
df.head()

After column selection: (140388, 8)


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Complaint Type,Borough,Status
0,60739073,03/31/2024 07:13:48 PM,04/09/2024 11:30:40 AM,EDC,Economic Development Corporation,Noise - Helicopter,MANHATTAN,Closed
1,60733861,03/31/2024 07:13:40 PM,03/31/2024 08:03:56 PM,NYPD,New York City Police Department,Illegal Parking,BRONX,Closed
2,60736342,03/31/2024 07:13:27 PM,04/01/2024 08:53:05 AM,TLC,Taxi and Limousine Commission,Lost Property,MANHATTAN,Closed
3,60739177,03/31/2024 07:13:22 PM,03/31/2024 07:28:24 PM,NYPD,New York City Police Department,Illegal Parking,BROOKLYN,Closed
4,60738322,03/31/2024 07:13:20 PM,03/31/2024 07:38:53 PM,NYPD,New York City Police Department,Illegal Parking,MANHATTAN,Closed


In [15]:
df = df.sample(n=5000, random_state=42).copy()
print("After sampling:", df.shape)


After sampling: (5000, 8)


In [16]:
# Drop rows missing critical graph fields
critical_cols = ["Unique Key", "Complaint Type", "Agency", "Borough", "Status"]
before = df.shape[0]
df = df.dropna(subset=critical_cols).copy()
after = df.shape[0]

print(f"Rows before dropna: {before}")
print(f"Rows after dropna : {after}")
print("Null counts after cleansing:\n", df.isnull().sum())


Rows before dropna: 5000
Rows after dropna : 5000
Null counts after cleansing:
 Unique Key         0
Created Date       0
Closed Date       62
Agency             0
Agency Name        0
Complaint Type     0
Borough            0
Status             0
dtype: int64


In [17]:
text_cols = ["Complaint Type", "Agency", "Agency Name", "Borough", "Status"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Standardize casing (keeps categories consistent)
df["Complaint Type"] = df["Complaint Type"].str.upper()
df["Agency"] = df["Agency"].str.upper()
df["Agency Name"] = df["Agency Name"].str.title()
df["Borough"] = df["Borough"].str.upper()
df["Status"] = df["Status"].str.upper()

df.head()


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Complaint Type,Borough,Status
38750,60699133,03/26/2024 06:59:15 PM,06/03/2024 01:09:38 PM,DOHMH,Department Of Health And Mental Hygiene,NON-RESIDENTIAL HEAT,MANHATTAN,CLOSED
118703,60612276,03/17/2024 11:56:28 PM,03/18/2024 12:30:29 AM,NYPD,New York City Police Department,NOISE - COMMERCIAL,MANHATTAN,CLOSED
72860,60658700,03/22/2024 08:17:51 PM,06/21/2024 08:33:34 PM,TLC,Taxi And Limousine Commission,FOR HIRE VEHICLE COMPLAINT,MANHATTAN,CLOSED
102818,60621523,03/19/2024 03:45:57 PM,03/19/2024 05:00:27 PM,NYPD,New York City Police Department,ILLEGAL PARKING,MANHATTAN,CLOSED
83624,60651260,03/21/2024 04:22:06 PM,03/05/2025 08:14:01 AM,DPR,Department Of Parks And Recreation,NEW TREE REQUEST,BROOKLYN,CLOSED


In [18]:
df["Created Date"] = pd.to_datetime(df["Created Date"], errors="coerce")
df["Closed Date"]  = pd.to_datetime(df["Closed Date"], errors="coerce")

print(df[["Created Date", "Closed Date"]].dtypes)
print("Nulls in Created Date:", df["Created Date"].isna().sum())


/tmp/ipython-input-3966423550.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Created Date"] = pd.to_datetime(df["Created Date"], errors="coerce")
/tmp/ipython-input-3966423550.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Closed Date"]  = pd.to_datetime(df["Closed Date"], errors="coerce")


Created Date    datetime64[ns]
Closed Date     datetime64[ns]
dtype: object
Nulls in Created Date: 0


In [19]:
df["created_year"] = df["Created Date"].dt.year
df["created_month"] = df["Created Date"].dt.month

# Resolution time in hours (only where Closed Date exists)
df["resolution_hours"] = (df["Closed Date"] - df["Created Date"]).dt.total_seconds() / 3600

df[["Unique Key","created_year","created_month","resolution_hours"]].head()


,Unique Key,created_year,created_month,resolution_hours
38750,60699133,2024,3,1650.173056
118703,60612276,2024,3,0.566944
72860,60658700,2024,3,2184.261944
102818,60621523,2024,3,1.241667
83624,60651260,2024,3,8367.865278


## 6. Node and Relationship Creation

### Objective
This step creates the graph structure in Neo4j by inserting nodes with appropriate labels and properties, and then defining relationships between them based on the NYC 311 dataset.

### Node Labels and Properties
- **Request**
  - `unique_key` (unique identifier)
  - `created_date`
  - `closed_date`
- **ComplaintType**
  - `name`
- **Agency**
  - `code` (Agency)
  - `name` (Agency Name)
- **Borough**
  - `name`
- **Status**
  - `name`

### Relationships
- `(Request)-[:HAS_COMPLAINT_TYPE]->(ComplaintType)`
- `(Request)-[:HANDLED_BY]->(Agency)`
- `(Request)-[:IN_BOROUGH]->(Borough)`
- `(Request)-[:HAS_STATUS]->(Status)`

### Approach
To avoid duplicates, `MERGE` is used when creating nodes. Constraints are created to enforce uniqueness. Data is inserted in batches using `UNWIND` for efficient loading.


In [22]:
def run_query(query, parameters=None):
    with driver.session() as session:
        return session.run(query, parameters or {}).data()

run_query("""
CALL db.labels() YIELD label
CALL {
  WITH label
  MATCH (n)
  WHERE label IN labels(n)
  RETURN label AS labelName, count(n) AS cnt
}
RETURN labelName, cnt
ORDER BY cnt DESC
""")

constraints = [
    "CREATE CONSTRAINT request_unique_key IF NOT EXISTS FOR (r:Request) REQUIRE r.unique_key IS UNIQUE",
    "CREATE CONSTRAINT complaint_name IF NOT EXISTS FOR (c:ComplaintType) REQUIRE c.name IS UNIQUE",
    "CREATE CONSTRAINT agency_code IF NOT EXISTS FOR (a:Agency) REQUIRE a.code IS UNIQUE",
    "CREATE CONSTRAINT borough_name IF NOT EXISTS FOR (b:Borough) REQUIRE b.name IS UNIQUE",
    "CREATE CONSTRAINT status_name IF NOT EXISTS FOR (s:Status) REQUIRE s.name IS UNIQUE",
]
for c in constraints:
    run_query(c)

print("Constraints created (or already exist).")

Constraints created (or already exist).


In [23]:
# Keep only the fields we need for graph insertion
df_load = df[[
    "Unique Key", "Created Date", "Closed Date",
    "Complaint Type", "Agency", "Agency Name",
    "Borough", "Status"
]].copy()

# Rename columns to simple keys for Neo4j
df_load = df_load.rename(columns={
    "Unique Key": "unique_key",
    "Created Date": "created_date",
    "Closed Date": "closed_date",
    "Complaint Type": "complaint_type",
    "Agency": "agency_code",
    "Agency Name": "agency_name",
    "Borough": "borough",
    "Status": "status"
})

# Convert datetimes to ISO strings (Neo4j-friendly) and handle NaT to None for both
df_load["created_date"] = df_load["created_date"].apply(
    lambda x: x.strftime("%Y-%m-%d %H:%M:%S") if pd.notna(x) else None
)
df_load["closed_date"]  = df_load["closed_date"].apply(
    lambda x: x.strftime("%Y-%m-%d %H:%M:%S") if pd.notna(x) else None
)

records = df_load.to_dict("records")
print("Records ready to load:", len(records))
records[0]

Records ready to load: 5000


{'unique_key': 60699133,
 'created_date': '2024-03-26 18:59:15',
 'closed_date': '2024-06-03 13:09:38',
 'complaint_type': 'NON-RESIDENTIAL HEAT',
 'agency_code': 'DOHMH',
 'agency_name': 'Department Of Health And Mental Hygiene',
 'borough': 'MANHATTAN',
 'status': 'CLOSED'}

In [24]:
cypher_load = """
UNWIND $rows AS row

// Nodes
MERGE (r:Request {unique_key: toInteger(row.unique_key)})
SET r.created_date = row.created_date,
    r.closed_date  = row.closed_date

MERGE (c:ComplaintType {name: row.complaint_type})
MERGE (a:Agency {code: row.agency_code})
SET a.name = row.agency_name

MERGE (b:Borough {name: row.borough})
MERGE (s:Status {name: row.status})

// Relationships
MERGE (r)-[:HAS_COMPLAINT_TYPE]->(c)
MERGE (r)-[:HANDLED_BY]->(a)
MERGE (r)-[:IN_BOROUGH]->(b)
MERGE (r)-[:HAS_STATUS]->(s)
"""

batch_size = 500
for i in range(0, len(records), batch_size):
    batch = records[i:i+batch_size]
    run_query(cypher_load, {"rows": batch})

print("Graph load complete.")


Graph load complete.


In [25]:
run_query("""
CALL db.labels() YIELD label
CALL {
  WITH label
  MATCH (n)
  WHERE label IN labels(n)
  RETURN label AS labelName, count(n) AS cnt
}
RETURN labelName, cnt
ORDER BY cnt DESC
""")


[{'labelName': 'Request', 'cnt': 9968},
 {'labelName': 'ComplaintType', 'cnt': 133},
 {'labelName': 'Agency', 'cnt': 13},
 {'labelName': 'Borough', 'cnt': 6},
 {'labelName': 'Status', 'cnt': 6},
 {'labelName': 'Student', 'cnt': 3},
 {'labelName': 'University', 'cnt': 3}]

In [26]:
run_query("""
CALL db.relationshipTypes() YIELD relationshipType
CALL {
  WITH relationshipType
  MATCH ()-[r]->()
  WHERE type(r) = relationshipType
  RETURN relationshipType AS relType, count(r) AS cnt
}
RETURN relType, cnt
ORDER BY cnt DESC
""")


[{'relType': 'HAS_COMPLAINT_TYPE', 'cnt': 9968},
 {'relType': 'HANDLED_BY', 'cnt': 9968},
 {'relType': 'IN_BOROUGH', 'cnt': 9968},
 {'relType': 'HAS_STATUS', 'cnt': 9968}]

In [27]:
run_query("""
MATCH (r:Request)-[:HAS_COMPLAINT_TYPE]->(c:ComplaintType),
      (r)-[:HANDLED_BY]->(a:Agency),
      (r)-[:IN_BOROUGH]->(b:Borough),
      (r)-[:HAS_STATUS]->(s:Status)
RETURN r, c, a, b, s
LIMIT 25
""")


[{'r': {'closed_date': '2024-02-06 11:46:00',
   'unique_key': 60228762,
   'created_date': '2024-02-04 09:07:00'},
  'c': {'name': 'DERELICT VEHICLES'},
  'a': {'code': 'DSNY', 'name': 'Department Of Sanitation'},
  'b': {'name': 'BRONX'},
  's': {'name': 'CLOSED'}},
 {'r': {'closed_date': '2024-06-03 13:10:57',
   'unique_key': 60184242,
   'created_date': '2024-01-30 15:15:46'},
  'c': {'name': 'NON-RESIDENTIAL HEAT'},
  'a': {'code': 'DOHMH', 'name': 'Department Of Health And Mental Hygiene'},
  'b': {'name': 'BRONX'},
  's': {'name': 'CLOSED'}},
 {'r': {'closed_date': '2024-01-08 16:48:13',
   'unique_key': 59966997,
   'created_date': '2024-01-08 15:57:14'},
  'c': {'name': 'ILLEGAL PARKING'},
  'a': {'code': 'NYPD', 'name': 'New York City Police Department'},
  'b': {'name': 'BRONX'},
  's': {'name': 'CLOSED'}},
 {'r': {'closed_date': '2024-02-27 11:48:16',
   'unique_key': 60433058,
   'created_date': '2024-02-27 05:41:48'},
  'c': {'name': 'ILLEGAL PARKING'},
  'a': {'code': '

## 7. Aggregation Operations (Graph Analytics)

### Objective
This step performs aggregation queries on the NYC 311 graph to summarize and analyze patterns in service requests. Aggregations are used to compute counts, distributions, and averages across complaint types, boroughs, agencies, and request statuses.

### Why Aggregations Matter
Aggregations provide measurable insights such as:
- which complaint categories dominate overall or within a borough
- which agencies receive the highest volume of requests
- how request statuses are distributed (open vs closed patterns)
- average resolution time for requests (when closed dates are available)

### Output
The following Cypher queries return aggregated results (counts and averages). Screenshots of query outputs are included in the final documentation as evidence of analysis.


Total requests by Borough

In [28]:
run_query("""
MATCH (r:Request)-[:IN_BOROUGH]->(b:Borough)
RETURN b.name AS borough, count(r) AS total_requests
ORDER BY total_requests DESC
""")


[{'borough': 'BROOKLYN', 'total_requests': 3069},
 {'borough': 'QUEENS', 'total_requests': 2510},
 {'borough': 'MANHATTAN', 'total_requests': 2056},
 {'borough': 'BRONX', 'total_requests': 1981},
 {'borough': 'STATEN ISLAND', 'total_requests': 344},
 {'borough': 'UNSPECIFIED', 'total_requests': 8}]

Top 10 complaint types overall

In [29]:
run_query("""
MATCH (r:Request)-[:HAS_COMPLAINT_TYPE]->(c:ComplaintType)
RETURN c.name AS complaint_type, count(r) AS total_requests
ORDER BY total_requests DESC
LIMIT 10
""")


[{'complaint_type': 'ILLEGAL PARKING', 'total_requests': 1588},
 {'complaint_type': 'HEAT/HOT WATER', 'total_requests': 1101},
 {'complaint_type': 'NOISE - RESIDENTIAL', 'total_requests': 904},
 {'complaint_type': 'BLOCKED DRIVEWAY', 'total_requests': 539},
 {'complaint_type': 'UNSANITARY CONDITION', 'total_requests': 339},
 {'complaint_type': 'STREET CONDITION', 'total_requests': 280},
 {'complaint_type': 'PLUMBING', 'total_requests': 240},
 {'complaint_type': 'ABANDONED VEHICLE', 'total_requests': 234},
 {'complaint_type': 'NOISE - COMMERCIAL', 'total_requests': 218},
 {'complaint_type': 'NOISE - STREET/SIDEWALK', 'total_requests': 209}]

Top complaint types per borough

In [30]:
run_query("""
MATCH (r:Request)-[:IN_BOROUGH]->(b:Borough),
      (r)-[:HAS_COMPLAINT_TYPE]->(c:ComplaintType)
RETURN b.name AS borough, c.name AS complaint_type, count(r) AS cnt
ORDER BY borough, cnt DESC
LIMIT 25
""")


[{'borough': 'BRONX', 'complaint_type': 'HEAT/HOT WATER', 'cnt': 402},
 {'borough': 'BRONX', 'complaint_type': 'NOISE - RESIDENTIAL', 'cnt': 229},
 {'borough': 'BRONX', 'complaint_type': 'ILLEGAL PARKING', 'cnt': 226},
 {'borough': 'BRONX', 'complaint_type': 'UNSANITARY CONDITION', 'cnt': 122},
 {'borough': 'BRONX', 'complaint_type': 'PAINT/PLASTER', 'cnt': 81},
 {'borough': 'BRONX', 'complaint_type': 'PLUMBING', 'cnt': 75},
 {'borough': 'BRONX', 'complaint_type': 'BLOCKED DRIVEWAY', 'cnt': 73},
 {'borough': 'BRONX', 'complaint_type': 'DOOR/WINDOW', 'cnt': 56},
 {'borough': 'BRONX', 'complaint_type': 'WATER LEAK', 'cnt': 53},
 {'borough': 'BRONX', 'complaint_type': 'GENERAL', 'cnt': 42},
 {'borough': 'BRONX', 'complaint_type': 'DERELICT VEHICLES', 'cnt': 42},
 {'borough': 'BRONX', 'complaint_type': 'STREET CONDITION', 'cnt': 37},
 {'borough': 'BRONX', 'complaint_type': 'NOISE - STREET/SIDEWALK', 'cnt': 35},
 {'borough': 'BRONX', 'complaint_type': 'FLOORING/STAIRS', 'cnt': 32},
 {'borou

Agency workload (requests handled by each agency)

In [31]:
run_query("""
MATCH (r:Request)-[:HANDLED_BY]->(a:Agency)
RETURN a.code AS agency_code, a.name AS agency_name, count(r) AS total_requests
ORDER BY total_requests DESC
LIMIT 10
""")


[{'agency_code': 'NYPD',
  'agency_name': 'New York City Police Department',
  'total_requests': 4127},
 {'agency_code': 'HPD',
  'agency_name': 'Department Of Housing Preservation And Development',
  'total_requests': 2565},
 {'agency_code': 'DSNY',
  'agency_name': 'Department Of Sanitation',
  'total_requests': 812},
 {'agency_code': 'DOT',
  'agency_name': 'Department Of Transportation',
  'total_requests': 743},
 {'agency_code': 'DEP',
  'agency_name': 'Department Of Environmental Protection',
  'total_requests': 507},
 {'agency_code': 'DOB',
  'agency_name': 'Department Of Buildings',
  'total_requests': 308},
 {'agency_code': 'DPR',
  'agency_name': 'Department Of Parks And Recreation',
  'total_requests': 273},
 {'agency_code': 'DOHMH',
  'agency_name': 'Department Of Health And Mental Hygiene',
  'total_requests': 246},
 {'agency_code': 'TLC',
  'agency_name': 'Taxi And Limousine Commission',
  'total_requests': 116},
 {'agency_code': 'EDC',
  'agency_name': 'Economic Developm

Status distribution (Open vs Closed etc.)

In [32]:
run_query("""
MATCH (r:Request)-[:HAS_STATUS]->(s:Status)
RETURN s.name AS status, count(r) AS total_requests
ORDER BY total_requests DESC
""")


[{'status': 'CLOSED', 'total_requests': 9875},
 {'status': 'IN PROGRESS', 'total_requests': 52},
 {'status': 'OPEN', 'total_requests': 17},
 {'status': 'PENDING', 'total_requests': 13},
 {'status': 'ASSIGNED', 'total_requests': 9},
 {'status': 'STARTED', 'total_requests': 2}]

Average resolution time (hours) by complaint type

In [33]:
run_query("""
MATCH (r:Request)-[:HAS_COMPLAINT_TYPE]->(c:ComplaintType)
WHERE r.closed_date IS NOT NULL AND r.created_date IS NOT NULL
WITH c.name AS complaint_type,
     duration.inSeconds(datetime(replace(r.created_date, ' ', 'T')), datetime(replace(r.closed_date, ' ', 'T'))).hours AS hrs
WHERE hrs IS NOT NULL AND hrs >= 0 AND hrs <= 24*365  // filter unrealistic values
RETURN complaint_type, round(avg(hrs), 2) AS avg_resolution_hours, count(*) AS closed_cases
ORDER BY avg_resolution_hours DESC
LIMIT 10
""")

[{'complaint_type': 'DAY CARE',
  'avg_resolution_hours': 7453.0,
  'closed_cases': 6},
 {'complaint_type': 'TATTOOING',
  'avg_resolution_hours': 7316.67,
  'closed_cases': 3},
 {'complaint_type': 'LOT CONDITION',
  'avg_resolution_hours': 7188.0,
  'closed_cases': 2},
 {'complaint_type': 'MOBILE FOOD VENDOR',
  'avg_resolution_hours': 7108.0,
  'closed_cases': 9},
 {'complaint_type': 'NEW TREE REQUEST',
  'avg_resolution_hours': 7072.61,
  'closed_cases': 28},
 {'complaint_type': 'DISPATCHED TAXI COMPLAINT',
  'avg_resolution_hours': 4584.0,
  'closed_cases': 1},
 {'complaint_type': 'SMOKING OR VAPING',
  'avg_resolution_hours': 3649.5,
  'closed_cases': 2},
 {'complaint_type': 'NON-RESIDENTIAL HEAT',
  'avg_resolution_hours': 2577.25,
  'closed_cases': 4},
 {'complaint_type': 'FOR HIRE VEHICLE COMPLAINT',
  'avg_resolution_hours': 2546.66,
  'closed_cases': 73},
 {'complaint_type': 'GREEN TAXI COMPLAINT',
  'avg_resolution_hours': 2512.0,
  'closed_cases': 1}]

## Querying the Graph Database

### Objective
This step demonstrates how the Neo4j graph database can be queried to retrieve connected information across requests, complaint types, agencies, boroughs, and statuses. Unlike table-based querying, graph queries directly traverse relationships to return meaningful connected results.

### Query Types Included
The following queries demonstrate:
- pattern-based traversal across multiple node types
- filtering by borough, status, and complaint type
- relationship-driven insights (which agencies handle which complaints where)
- example request “paths” through the graph



Show connected data for sample requests

In [34]:
run_query("""
MATCH (r:Request)-[:HAS_COMPLAINT_TYPE]->(c:ComplaintType),
      (r)-[:HANDLED_BY]->(a:Agency),
      (r)-[:IN_BOROUGH]->(b:Borough),
      (r)-[:HAS_STATUS]->(s:Status)
RETURN r.unique_key AS request_id,
       c.name AS complaint_type,
       a.code AS agency_code,
       b.name AS borough,
       s.name AS status
LIMIT 25
""")


[{'request_id': 60228762,
  'complaint_type': 'DERELICT VEHICLES',
  'agency_code': 'DSNY',
  'borough': 'BRONX',
  'status': 'CLOSED'},
 {'request_id': 60184242,
  'complaint_type': 'NON-RESIDENTIAL HEAT',
  'agency_code': 'DOHMH',
  'borough': 'BRONX',
  'status': 'CLOSED'},
 {'request_id': 59966997,
  'complaint_type': 'ILLEGAL PARKING',
  'agency_code': 'NYPD',
  'borough': 'BRONX',
  'status': 'CLOSED'},
 {'request_id': 60433058,
  'complaint_type': 'ILLEGAL PARKING',
  'agency_code': 'NYPD',
  'borough': 'BRONX',
  'status': 'CLOSED'},
 {'request_id': 60263700,
  'complaint_type': 'CONSUMER COMPLAINT',
  'agency_code': 'DCWP',
  'borough': 'BRONX',
  'status': 'CLOSED'},
 {'request_id': 59938567,
  'complaint_type': 'ABANDONED VEHICLE',
  'agency_code': 'NYPD',
  'borough': 'BRONX',
  'status': 'CLOSED'},
 {'request_id': 60184336,
  'complaint_type': 'STREET SIGN - MISSING',
  'agency_code': 'DOT',
  'borough': 'BRONX',
  'status': 'CLOSED'},
 {'request_id': 59940098,
  'complain

Find Open requests in a specific borough

In [35]:
run_query("""
MATCH (r:Request)-[:IN_BOROUGH]->(b:Borough),
      (r)-[:HAS_STATUS]->(s:Status),
      (r)-[:HAS_COMPLAINT_TYPE]->(c:ComplaintType)
WHERE b.name = "BROOKLYN" AND s.name = "OPEN"
RETURN r.unique_key AS request_id, c.name AS complaint_type, s.name AS status, b.name AS borough
LIMIT 25
""")


[{'request_id': 60408767,
  'complaint_type': 'GRAFFITI',
  'status': 'OPEN',
  'borough': 'BROOKLYN'},
 {'request_id': 60450300,
  'complaint_type': 'PAINT/PLASTER',
  'status': 'OPEN',
  'borough': 'BROOKLYN'},
 {'request_id': 60150008,
  'complaint_type': 'STREET CONDITION',
  'status': 'OPEN',
  'borough': 'BROOKLYN'},
 {'request_id': 60479576,
  'complaint_type': 'SEWER',
  'status': 'OPEN',
  'borough': 'BROOKLYN'},
 {'request_id': 60616657,
  'complaint_type': 'OUTSIDE BUILDING',
  'status': 'OPEN',
  'borough': 'BROOKLYN'},
 {'request_id': 60608964,
  'complaint_type': 'STREET CONDITION',
  'status': 'OPEN',
  'borough': 'BROOKLYN'},
 {'request_id': 60629718,
  'complaint_type': 'GENERAL CONSTRUCTION/PLUMBING',
  'status': 'OPEN',
  'borough': 'BROOKLYN'}]

Which agencies handle a specific complaint type?

In [36]:
run_query("""
MATCH (r:Request)-[:HAS_COMPLAINT_TYPE]->(c:ComplaintType),
      (r)-[:HANDLED_BY]->(a:Agency)
WHERE c.name = "ILLEGAL PARKING"
RETURN DISTINCT a.code AS agency_code, a.name AS agency_name
LIMIT 25
""")


[{'agency_code': 'NYPD', 'agency_name': 'New York City Police Department'}]

Complaint type → boroughs where it appears most

In [37]:
run_query("""
MATCH (r:Request)-[:HAS_COMPLAINT_TYPE]->(c:ComplaintType),
      (r)-[:IN_BOROUGH]->(b:Borough)
WHERE c.name = "ILLEGAL PARKING"
RETURN b.name AS borough, count(r) AS requests
ORDER BY requests DESC
LIMIT 10
""")


[{'borough': 'BROOKLYN', 'requests': 607},
 {'borough': 'QUEENS', 'requests': 512},
 {'borough': 'BRONX', 'requests': 226},
 {'borough': 'MANHATTAN', 'requests': 195},
 {'borough': 'STATEN ISLAND', 'requests': 48}]

In [38]:
run_query("""
MATCH p = (r:Request)-[:HAS_COMPLAINT_TYPE|:HANDLED_BY|:IN_BOROUGH|:HAS_STATUS]->()
RETURN p
LIMIT 10
""")


[{'p': [{'closed_date': '2024-01-06 02:40:00',
    'unique_key': 59934952,
    'created_date': '2024-01-05 16:26:00'},
   'HAS_COMPLAINT_TYPE',
   {'name': 'WATER SYSTEM'}]},
 {'p': [{'closed_date': '2024-01-06 02:40:00',
    'unique_key': 59934952,
    'created_date': '2024-01-05 16:26:00'},
   'HANDLED_BY',
   {'code': 'DEP', 'name': 'Department Of Environmental Protection'}]},
 {'p': [{'closed_date': '2024-01-06 02:40:00',
    'unique_key': 59934952,
    'created_date': '2024-01-05 16:26:00'},
   'IN_BOROUGH',
   {'name': 'BROOKLYN'}]},
 {'p': [{'closed_date': '2024-01-06 02:40:00',
    'unique_key': 59934952,
    'created_date': '2024-01-05 16:26:00'},
   'HAS_STATUS',
   {'name': 'CLOSED'}]},
 {'p': [{'closed_date': '2024-02-06 11:46:00',
    'unique_key': 60228762,
    'created_date': '2024-02-04 09:07:00'},
   'HAS_COMPLAINT_TYPE',
   {'name': 'DERELICT VEHICLES'}]},
 {'p': [{'closed_date': '2024-02-06 11:46:00',
    'unique_key': 60228762,
    'created_date': '2024-02-04 09:07:

## Key Insights

Based on the graph-based analysis of the NYC 311 service request data, several important insights were identified:

- **Complaint concentration varies significantly by borough**, indicating that certain types of issues are more prevalent in specific geographic areas. This highlights the importance of location-aware service planning.

- **A small number of agencies handle the majority of service requests**, with agencies such as the NYPD receiving a disproportionately high volume of complaints related to enforcement and public order issues.

- **Complaint types such as illegal parking and noise-related issues appear most frequently**, demonstrating recurring urban concerns that may require targeted policy or operational responses.

- **Request status distribution reveals a clear separation between resolved and unresolved cases**, which can be used to identify backlog patterns and opportunities for improving response efficiency.

- **Average resolution time differs by complaint type**, suggesting that some issues are inherently more complex or resource-intensive to resolve than others.

The graph database made it easy to uncover these patterns by directly traversing relationships between requests, agencies, complaint types, and locations without complex join operations.


## Conclusion

This project demonstrated the effectiveness of using a graph database to analyze NYC 311 service request data. By modeling service requests, agencies, complaint types, boroughs, and statuses as interconnected nodes, the graph structure provided an intuitive and flexible way to explore complex relationships within the data.

Compared to traditional relational databases, the graph database enabled more efficient querying and clearer insight discovery through relationship-based traversal. Tasks such as identifying high-volume complaint types, analyzing agency workload, and examining geographic trends were simplified and made more transparent.

Overall, this project shows that graph databases are well suited for urban service data that involves many interconnected entities. The approach used in this project can be extended to larger datasets or additional node types in the future to support deeper analysis, improved service planning, and more informed decision-making.
